# Day 171 — LangChain Document Loaders + LCEL
## Month 10, Day 3 | Google Colab | Groq Free API

---

### Month 10 Progress
| Day | Topic | Score |
|-----|-------|-------|
| 169 | LangChain Chains & Memory | ✅ 80/80+10★ |
| 170 | LangChain Tools & Agents | ✅ 80/80+10★ |
| **171** | **Document Loaders + LCEL** | **← Today** |
| 172 | LangChain Capstone | Upcoming |

---

### Today's Scorecard
| Task | Topic | Points |
|------|-------|--------|
| T1 | CSVLoader — Load ReviewPulse as Documents | 15 |
| T2 | LCEL Basic Chain — `|` operator + StrOutputParser | 15 |
| T3 | RunnableParallel — Two-Branch Analysis | 20 |
| T4 | RunnablePassthrough — Document-Grounded QA Chain | 15 |
| T5 | RunnableLambda + Full LCEL Pipeline + 3-Bullet NRA | 15 |
| ★ | `.batch()` — Process Multiple Inputs Efficiently | 10★ |
| **Total** | | **80/80 + 10★** |

**Dataset:** ReviewPulse India (600 rows, seed=155)  
**LLM:** Groq free API → `llama-3.1-8b-instant`  
**Environment:** Google Colab (CPU fine)

---

### Why this matters for freelance
Day 169 = fixed chains. Day 170 = agents that pick tools.  
Day 171 = **how data enters the LLM in the first place.**

Every real client project starts with a document: a CSV, a PDF, a log file, a knowledge base.  
Document Loaders turn those files into `Document` objects that LangChain can reason over.  
LCEL (LangChain Expression Language) is the modern way to compose those reasoning steps — faster,  
more readable, and production-safe compared to the old `LLMChain` pattern.  

Together: **load data → compose reasoning → answer business questions.** That's the full loop.

---
## ⚙️ CELL 1 — Install & Restart
**RUN THIS FIRST → Runtime → Restart → then run all remaining cells**

In [ ]:
# PINNED VERSIONS — do NOT change. LangChain 0.3+ broke LLMChain and Memory imports.
!pip install -q \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-groq==0.1.9 \
    langchain-core==0.2.38

print("✅ Install complete — Runtime → Restart → then continue")

✅ Install complete — Runtime → Restart → then continue


---
## 📦 CELL 2 — Imports & API Key
After restart: run this cell first.

In [ ]:
import os
import pandas as pd
import numpy as np

# Document Loaders
from langchain_community.document_loaders import CSVLoader

# LCEL core components
from langchain_core.runnables import (
    RunnablePassthrough,
    RunnableParallel,
    RunnableLambda
)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# LLM
from langchain_groq import ChatGroq

# Groq API key
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Initialize LLM — used throughout
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,   # deterministic
    max_tokens=512
)

print("✅ Imports complete")
print(f"LLM model: {llm.model_name}")

✅ Imports complete
LLM model: llama-3.1-8b-instant


---
## 🗂️ RAW DATA — DO NOT MODIFY
ReviewPulse India | 600 rows | seed=155

In [ ]:
# ─── RAW DATA CELL — DO NOT MODIFY ANYTHING BELOW THIS LINE ───────────────────
np.random.seed(155)

n = 600
categories   = ['Web Development', 'Data Analysis', 'Content Writing', 'Design', 'SEO']
sentiments   = ['positive', 'negative', 'neutral']
review_pool  = [
    "Excellent work, very professional and delivered on time.",
    "Poor quality, did not meet expectations at all.",
    "Average work, nothing special but completed the task.",
    "Great communication and outstanding results.",
    "Missed deadlines repeatedly and quality was subpar.",
    "Satisfactory output, would consider hiring again.",
    "Top-notch skills, exceeded all requirements.",
    "Very disappointing, wasted time and money.",
    "Decent work for the price, no major complaints.",
    "Highly recommend, will definitely hire again."
]

raw_df = pd.DataFrame({
    'review_id'   : range(1, n+1),
    'reviewer'    : [f'Client_{i:03d}' for i in range(1, n+1)],
    'category'    : np.random.choice(categories, n),
    'rating'      : np.random.choice([1,2,3,4,5], n, p=[0.20,0.25,0.15,0.20,0.20]),
    'sentiment'   : np.random.choice(sentiments, n, p=[0.257, 0.443, 0.300]),
    'review_text' : np.random.choice(review_pool, n),
    'hired_again' : np.random.choice([0,1], n, p=[0.637, 0.363])
})

# ─── SAVE CSV (needed by CSVLoader) ───────────────────────────────────────────
CSV_PATH = "/content/reviewpulse_india.csv"
raw_df.to_csv(CSV_PATH, index=False)

print(f"Dataset shape : {raw_df.shape}")
print(f"Columns       : {list(raw_df.columns)}")
print(f"CSV saved to  : {CSV_PATH}")
print("─" * 50)
print(raw_df.head(3))
# ─── DO NOT MODIFY ABOVE ───────────────────────────────────────────────────────

Dataset shape : (600, 7)
Columns       : ['review_id', 'reviewer', 'category', 'rating', 'sentiment', 'review_text', 'hired_again']
CSV saved to  : /content/reviewpulse_india.csv
──────────────────────────────────────────────────
   review_id    reviewer         category  rating sentiment  \
0          1  Client_001              SEO       4   neutral   
1          2  Client_002    Data Analysis       3  positive   
2          3  Client_003  Web Development       5  positive   

                                         review_text  hired_again  
0  Satisfactory output, would consider hiring again.            1  
1         Very disappointing, wasted time and money.            0  
2  Missed deadlines repeatedly and quality was su...            0  


---
## 📘 CONCEPT NOTES

### 1. What is a Document?
In LangChain, a `Document` is a simple object with two fields:
- `page_content` — the text the LLM will read
- `metadata` — a dict of extra info (filename, row number, source URL, etc.)

```python
from langchain_core.documents import Document
doc = Document(page_content="Great work!", metadata={"source": "review_42", "row": 42})
```

### 2. CSVLoader
Converts each CSV row into one `Document`.  
`page_content` = all column values joined as `column: value\n...`  
`metadata` = `{"source": "filepath", "row": row_index}`

```python
from langchain_community.document_loaders import CSVLoader
loader = CSVLoader(file_path="data.csv")   # one doc per row
docs = loader.load()
```

### 3. LCEL — LangChain Expression Language
LCEL uses Python's `|` operator to compose Runnables into chains.
Every LCEL component implements `.invoke()`, `.batch()`, `.stream()`.

**Old way (Day 169):**
```python
chain = LLMChain(llm=llm, prompt=prompt)   # verbose, lots of setup
```

**LCEL way (today):**
```python
chain = prompt | llm | StrOutputParser()   # composable, readable
```

### 4. Core LCEL Runnables

| Runnable | What it does |
|---|---|
| `RunnablePassthrough()` | Passes input unchanged to the next step |
| `RunnablePassthrough.assign(key=fn)` | Passes input AND adds a new key from `fn(input)` |
| `RunnableParallel({...})` | Runs multiple chains on the same input simultaneously |
| `RunnableLambda(fn)` | Wraps any Python function as a chain step |
| `StrOutputParser()` | Extracts the string text from an LLM response object |

### 5. LCEL Data Flow
```
Input dict
    ↓
RunnablePassthrough / RunnableLambda  ← pre-processing / enrichment
    ↓
ChatPromptTemplate                    ← builds the prompt text
    ↓
ChatGroq (LLM)                        ← calls the API
    ↓
StrOutputParser()                     ← extracts plain text string
    ↓
Final answer (str)
```

### 6. .batch() vs .invoke()
- `.invoke(input)` — one input, one output, sequential
- `.batch([input1, input2, ...])` — list of inputs, processed efficiently (can run in threads)
- In production: use `.batch()` when you have many similar inputs to process

---
## ✏️ TASK 1 — CSVLoader: Load ReviewPulse as Documents [15 pts]

**Objective:** Load the saved CSV using `CSVLoader`. Inspect the Document structure.  
Then write one NRA insight about what CSVLoader produced.

### Sub-tasks
- **T1a (4 pts):** Create a `CSVLoader` with `file_path=CSV_PATH` and call `.load()` → store as `docs`
- **T1b (4 pts):** Print: `len(docs)`, type of `docs[0]`, `docs[0].page_content[:300]`, `docs[0].metadata`
- **T1c (4 pts):** Print the `page_content` of `docs[5]` (row 5, zero-indexed)
- **T1d (3 pts):** NRA — one bullet using the printed `len(docs)` value

**NRA Rule:** Your number in T1d must match the printed `len(docs)` output exactly.  
Do NOT type it from memory.

In [ ]:
# --------------------------------------------------------------------
# TASK 1: Load the saved CSV using CSVLoader and inspect Document structure.
# GOAL: Understand how CSVLoader converts each row into a Document.
# METHOD: Instantiate CSVLoader with file_path, call .load() to get a list
#         of Documents. Print total count, metadata, and page_content of
#         specific rows. Then write an NRA insight based on the printed count.
# --------------------------------------------------------------------

# T1a – Load CSV as Documents
loader = CSVLoader(file_path=CSV_PATH)
docs = loader.load()          # list of Document objects, one per row

# T1b – Inspect structure
print(f"Total documents loaded : {len(docs)}")
print(f"Type of docs[0]        : {type(docs[0])}")   # should be langchain_core.documents.Document
print("\n--- docs[0].page_content (first 300 chars) ---")
print(docs[0].page_content[:300])
print("\n--- docs[0].metadata ---")
print(docs[0].metadata)       # contains source file path and row number

# T1c – Row 5 (zero‑indexed)
print("\n--- docs[5].page_content ---")
print(docs[5].page_content)

Total documents loaded : 600
Type of docs[0]        : <class 'langchain_core.documents.base.Document'>

--- docs[0].page_content (first 300 chars) ---
review_id: 1
reviewer: Client_001
category: SEO
rating: 4
sentiment: neutral
review_text: Satisfactory output, would consider hiring again.
hired_again: 1

--- docs[0].metadata ---
{'source': '/content/reviewpulse_india.csv', 'row': 0}

--- docs[5].page_content ---
review_id: 6
reviewer: Client_006
category: Design
rating: 2
sentiment: positive
review_text: Missed deadlines repeatedly and quality was subpar.
hired_again: 0


In [ ]:
# T1d – NRA Insight (fill after reading printed len(docs))
nra_t1 = """
Number : 600 documents loaded.
Reason : CSVLoader maps each CSV row to one Document object because each row is an
         independent data record; combining rows into one Document would lose row‑level
         metadata (source, row index) needed for tracing answers back to source records.
Action : Use CSVLoader over TextLoader whenever row‑level retrieval or filtering by
         metadata is required in the downstream chain.
"""
print(nra_t1)


Number : 600 documents loaded.
Reason : CSVLoader maps each CSV row to one Document object because each row is an
         independent data record; combining rows into one Document would lose row‑level
         metadata (source, row index) needed for tracing answers back to source records.
Action : Use CSVLoader over TextLoader whenever row‑level retrieval or filtering by
         metadata is required in the downstream chain.



---
## ✏️ TASK 2 — LCEL Basic Chain with `|` Operator [15 pts]

**Objective:** Build your first LCEL chain using `ChatPromptTemplate | ChatGroq | StrOutputParser`.  
Compare the LCEL output to what `LLMChain` would have needed.

### Sub-tasks
- **T2a (4 pts):** Create a `ChatPromptTemplate.from_template()` that accepts `{question}` and `{context}`.  
  The template should ask the LLM to answer the question based on the provided context.
- **T2b (4 pts):** Build the LCEL chain: `prompt_t2 | llm | StrOutputParser()`  
  Store as `basic_chain`
- **T2c (4 pts):** Invoke the chain with:  
  `question = "What is the overall sentiment distribution of these freelancer reviews?"`  
  `context = docs[0].page_content + "\n" + docs[1].page_content + "\n" + docs[2].page_content`  
  Print the result string.
- **T2d (3 pts):** Print `type(basic_chain)` — note the LCEL chain type vs old LLMChain

**Key insight to notice:** LCEL chains have type `RunnableSequence`, not `LLMChain`.

In [ ]:
# --------------------------------------------------------------------
# TASK 2: Build your first LCEL chain using the | operator.
# GOAL: Create a chain that answers a question based on a provided context.
# METHOD: Define a ChatPromptTemplate with {question} and {context},
#         pipe it to the LLM and then to StrOutputParser.
#         Invoke with a sample context from the first 3 documents.
#         Print the result and the chain type.
# --------------------------------------------------------------------

# T2a – Build prompt template
prompt_t2 = ChatPromptTemplate.from_template(
    """
    You are a data analyst reviewing freelancer feedback.
    Context:
    {context}

    Question: {question}

    Answer concisely in 2-3 sentences.
    """
)

# T2b – Build LCEL chain
basic_chain = prompt_t2 | llm | StrOutputParser()

# T2c – Invoke the chain
context_sample = docs[0].page_content + "\n" + docs[1].page_content + "\n" + docs[2].page_content

result_t2 = basic_chain.invoke({
    "question": "What is the overall sentiment distribution of these freelancer reviews?",
    "context" : context_sample
})

print("=== LCEL Basic Chain Output ===")
print(result_t2)
print()

# T2d – Chain type
print(f"Chain type : {type(basic_chain)}")   # <class 'langchain_core.runnables.base.RunnableSequence'>

=== LCEL Basic Chain Output ===
The overall sentiment distribution of these freelancer reviews is as follows: 

- Neutral sentiment: 1 review (33.3% of total reviews)
- Positive sentiment: 1 review (33.3% of total reviews)
- Negative sentiment: 1 review (33.3% of total reviews)

Note: The neutral sentiment review has a positive rating, which may seem contradictory, but the sentiment is neutral due to the reviewer's mixed comments.

Chain type : <class 'langchain_core.runnables.base.RunnableSequence'>


---
## ✏️ TASK 3 — RunnableParallel: Two-Branch Analysis [20 pts]

**Objective:** Run two different analysis prompts on the same input simultaneously  
using `RunnableParallel`. Then write one NRA bullet from the printed output.

### Background
`RunnableParallel` runs multiple chains on the **same input** at the same time  
and returns a dict with all results.

```python
parallel = RunnableParallel({
    "branch_a": chain_a,
    "branch_b": chain_b
})
result = parallel.invoke({"input": ...})   # result is a dict
print(result["branch_a"])   # output of chain_a
print(result["branch_b"])   # output of chain_b
```

### Sub-tasks
- **T3a (5 pts):** Build `context_10` — page_content of the first 10 documents joined by `"\n\n"`
- **T3b (5 pts):** Build two separate LCEL chains:  
  - `chain_problems` — asks: *"List the top 2 problems mentioned in these reviews."*  
  - `chain_strengths` — asks: *"List the top 2 strengths mentioned in these reviews."*  
  Both take `{context}` as input.
- **T3c (5 pts):** Create `parallel_chain = RunnableParallel({"problems": chain_problems, "strengths": chain_strengths})`  
  Invoke with `{"context": context_10}`. Print both outputs.
- **T3d (5 pts):** NRA — one bullet comparing the problems vs strengths output.  
  The Number must come from the printed result (e.g., count of items listed, or quote a specific phrase).

**NRA Rule for T3d:** If the LLM listed items as "1. ..., 2. ..." count them and use that count as the Number.

In [ ]:
# --------------------------------------------------------------------
# TASK 3: Run two analysis branches in parallel on the same input.
# GOAL: Demonstrate how RunnableParallel processes multiple chains simultaneously.
# METHOD: Build context from first 10 documents. Create two separate LCEL chains:
#         one for problems, one for strengths. Wrap them in RunnableParallel.
#         Invoke once and print both outputs. Write NRA from the results.
# --------------------------------------------------------------------

# T3a – Build 10‑document context
context_10 = "\n\n".join([doc.page_content for doc in docs[:10]])
print(f"Context_10 built from {len(docs[:10])} documents")
print(f"Total characters in context: {len(context_10)}")

Context_10 built from 10 documents
Total characters in context: 1615


In [ ]:
# T3b – Build two single‑input prompt templates
prompt_problems = ChatPromptTemplate.from_template(
    "Based on these freelancer reviews, list the top 2 problems mentioned.\n\n"
    "Reviews:\n{context}\n\n"
    "Format: numbered list, one sentence each."
)

prompt_strengths = ChatPromptTemplate.from_template(
    "Based on these freelancer reviews, list the top 2 strengths mentioned.\n\n"
    "Reviews:\n{context}\n\n"
    "Format: numbered list, one sentence each."
)

parser = StrOutputParser()

# Build the two chains
chain_problems  = prompt_problems | llm | parser
chain_strengths = prompt_strengths | llm | parser

print("✅ Two chains built")

✅ Two chains built


In [ ]:
# T3c – RunnableParallel: run both chains simultaneously
parallel_chain = RunnableParallel({
    "problems": chain_problems,
    "strengths": chain_strengths
})

result_t3 = parallel_chain.invoke({"context": context_10})

print("=== PROBLEMS BRANCH ===")
print(result_t3["problems"])
print()
print("=== STRENGTHS BRANCH ===")
print(result_t3["strengths"])

=== PROBLEMS BRANCH ===
Based on the freelancer reviews, the top 2 problems mentioned are:

1. Missed deadlines repeatedly, which was mentioned in reviews 3, 6, and implied in review 7 (despite the reviewer highly recommending the freelancer).
2. Poor quality or subpar work, which was mentioned in reviews 3, 4, and 9 (despite the reviewer having no major complaints).

=== STRENGTHS BRANCH ===
Based on the freelancer reviews, the top 2 strengths mentioned are:

1. Great communication skills, as mentioned in review_id: 8, where the client praised the freelancer's communication despite having negative sentiments about the overall work.
2. Top-notch skills and ability to exceed requirements, as mentioned in review_id: 5, where the client was highly satisfied with the freelancer's work in data analysis.


In [ ]:
# T3d – NRA from printed output
nra_t3 = """
Number : 2 problems identified: missed deadlines and poor quality of work.
Reason : Negative reviews cluster around delivery failures because client expectations
         on timeline and deliverables are rarely set in writing at the start.
Action : Add a project kickoff checklist with clear timeline milestones to every
         new client contract to reduce deadline disputes.
"""
print(nra_t3)


Number : 2 problems identified: missed deadlines and poor quality of work.
Reason : Negative reviews cluster around delivery failures because client expectations
         on timeline and deliverables are rarely set in writing at the start.
Action : Add a project kickoff checklist with clear timeline milestones to every
         new client contract to reduce deadline disputes.



---
## ✏️ TASK 4 — RunnablePassthrough: Document-Grounded QA Chain [15 pts]

**Objective:** Use `RunnablePassthrough` to build a document-grounded QA chain where  
the original question passes through unchanged while a retrieval step adds context.

### How RunnablePassthrough.assign() works
```python
# Input: {"question": "How many reviews?"}
#
# RunnablePassthrough.assign(context=retriever_fn)
# → runs retriever_fn({"question": "How many reviews?"}) to get context string
# → output: {"question": "How many reviews?", "context": "<retrieved text>"}
#
# This enriched dict then flows into the prompt template
```

### Sub-tasks
- **T4a (4 pts):** Build `retrieve_fn` — a Python function that takes a dict with key `"question"`  
  and returns the `page_content` of the first 5 documents joined by `"\n---\n"`  
  *(In a real RAG system this would be a vector search; today we use the first 5 for simplicity.)*
- **T4b (4 pts):** Build the full LCEL chain:  
  ```python
  qa_chain = (
      RunnablePassthrough.assign(context=RunnableLambda(retrieve_fn))
      | prompt_qa
      | llm
      | StrOutputParser()
  )
  ```
  `prompt_qa` takes `{question}` and `{context}`.
- **T4c (4 pts):** Invoke with `{"question": "What categories of work do these freelancers do?"}`.  
  Print the answer.
- **T4d (3 pts):** Print `type(qa_chain)` and explain in a one-line comment why it's `RunnableSequence`.

In [ ]:
# --------------------------------------------------------------------
# TASK 4: Build a QA chain that enriches the input with a context retrieval step.
# GOAL: Use RunnablePassthrough.assign() to add a "context" key from a custom
#       retrieval function, then feed that to the prompt.
# METHOD: Define a retrieve_fn that returns page_content of first 5 docs.
#         Build the chain with .assign(context=RunnableLambda(retrieve_fn)),
#         then pipe to prompt, LLM, and parser. Invoke with a question.
# --------------------------------------------------------------------

# T4a – Define the retrieval function (simulated – no vector DB needed today)
def retrieve_fn(input_dict):
    """
    Returns the first 5 documents' page_content joined by separator.
    In a real RAG pipeline this would be replaced by vector similarity search.
    """
    first_five = docs[:5]
    return "\n---\n".join([doc.page_content for doc in first_five])

# Quick test
test_context = retrieve_fn({"question": "test"})
print(f"retrieve_fn output length : {len(test_context)} chars")
print(f"Preview:\n{test_context[:200]}")

retrieve_fn output length : 812 chars
Preview:
review_id: 1
reviewer: Client_001
category: SEO
rating: 4
sentiment: neutral
review_text: Satisfactory output, would consider hiring again.
hired_again: 1
---
review_id: 2
reviewer: Client_002
categor


In [ ]:
# T4b – Build the QA prompt + full LCEL chain
prompt_qa = ChatPromptTemplate.from_template(
    "You are a business analyst reviewing freelancer feedback.\n"
    "Use ONLY the context below to answer.\n\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n\n"
    "Answer in 2-3 sentences. Be specific."
)

qa_chain = (
    RunnablePassthrough.assign(context=RunnableLambda(retrieve_fn))
    | prompt_qa
    | llm
    | StrOutputParser()
)

print("✅ QA chain built")

✅ QA chain built


In [ ]:
# T4c – Invoke and print answer
result_t4 = qa_chain.invoke({"question": "What categories of work do these freelancers do?"})

print("=== QA Chain Answer ===")
print(result_t4)
print()

# T4d – Chain type
print(f"Chain type: {type(qa_chain)}")
# One-line comment explaining why it's RunnableSequence:
# RunnablePassthrough, prompt, LLM, and parser are all Runnables; chaining them
# with | creates a RunnableSequence that executes each step in order.

=== QA Chain Answer ===
Based on the provided feedback, the freelancers offer services in the following categories: SEO, Data Analysis, and Web Development. These categories are explicitly mentioned in the reviews, indicating the types of work the freelancers specialize in.

Chain type: <class 'langchain_core.runnables.base.RunnableSequence'>


---
## ✏️ TASK 5 — RunnableLambda + Full LCEL Pipeline + 3-Bullet NRA [15 pts]

**Objective:** Use `RunnableLambda` to add a custom pre-processing step that extracts  
only the `review_text` column values from the documents before passing to the LLM.  
Then run a 3-step analysis and produce a full 3-bullet NRA report.

### Why RunnableLambda?
Sometimes you need a custom Python transformation in the middle of a chain —  
not a prompt, not an LLM call. `RunnableLambda(fn)` wraps any `fn` as a chain step.

### Sub-tasks
- **T5a (4 pts):** Define `extract_review_texts(input_dict)`:  
  - Input: `{"docs": [Document, ...], "question": "..."}`
  - Parse each `doc.page_content` to find the line containing `review_text:`  
  - Return a dict: `{"review_texts": "<joined texts>", "question": input_dict["question"]}`  
  - Hint: split `page_content` by `"\n"`, find the line starting with `review_text:`, strip that prefix
- **T5b (5 pts):** Build the full LCEL pipeline:
  ```
  RunnableLambda(extract_review_texts) | prompt_t5 | llm | StrOutputParser()
  ```
  `prompt_t5` takes `{review_texts}` and `{question}`
- **T5c (6 pts):** Run 3 invocations (use first 20 docs each time):  
  - Q1: *"What emotion dominates these freelancer reviews?"*  
  - Q2: *"What is the single most common complaint?"*  
  - Q3: *"Based on these reviews, what should a freelancer do differently?"*  
  Print all 3 answers. Then write the 3-bullet NRA below.

**NRA Rule for T5c:** Each bullet's Number must be read from what printed in the cell above.  
Do not estimate or assume — run the cell, then write.

In [ ]:
# --------------------------------------------------------------------
# TASK 5: Use RunnableLambda to add custom preprocessing that extracts
#         only the review_text column from documents.
# GOAL: Build a full pipeline that transforms the input, then runs an LLM.
# METHOD: Define extract_review_texts to parse page_content and collect
#         review_text lines. Wrap it in RunnableLambda, pipe to prompt,
#         LLM, and parser. Run 3 questions on the first 20 docs and print.
#         Write a 3‑bullet NRA from the printed answers.
# --------------------------------------------------------------------

# T5a – Extract review_text lines from documents
def extract_review_texts(input_dict):
    """
    Parses Document.page_content lines to extract review_text values.
    CSVLoader formats each field as 'column_name: value\\n'.
    """
    docs_in   = input_dict["docs"]
    question  = input_dict["question"]
    texts     = []

    for doc in docs_in:
        for line in doc.page_content.split("\n"):
            if line.startswith("review_text:"):
                # extract value after 'review_text: '
                texts.append(line.replace("review_text:", "").strip())
                break

    joined = "\n".join([f"{i+1}. {t}" for i, t in enumerate(texts)])

    return {"review_texts": joined, "question": question}

# Quick test with 3 docs
test_out = extract_review_texts({"docs": docs[:3], "question": "test"})
print("--- Extract function test ---")
print(test_out["review_texts"])

--- Extract function test ---
1. Satisfactory output, would consider hiring again.
2. Very disappointing, wasted time and money.
3. Missed deadlines repeatedly and quality was subpar.


In [ ]:
# T5b – Build full LCEL pipeline
prompt_t5 = ChatPromptTemplate.from_template(
    "You are analysing freelancer review texts.\n\n"
    "Reviews:\n{review_texts}\n\n"
    "Question: {question}\n\n"
    "Answer in 2-3 sentences. Be specific. Mention at least one concrete example."
)

full_pipeline = (
    RunnableLambda(extract_review_texts)
    | prompt_t5
    | llm
    | StrOutputParser()
)

print(f"Pipeline type: {type(full_pipeline)}")   # RunnableSequence

Pipeline type: <class 'langchain_core.runnables.base.RunnableSequence'>


In [ ]:
# T5c – Run 3 questions and collect answers
docs_20 = docs[:20]  # first 20 documents

ans1 = full_pipeline.invoke({"docs": docs_20, "question": "What emotion dominates these freelancer reviews?"})
ans2 = full_pipeline.invoke({"docs": docs_20, "question": "What is the single most common complaint?"})
ans3 = full_pipeline.invoke({"docs": docs_20, "question": "Based on these reviews, what should a freelancer do differently?"})

print("=== Q1: Dominant Emotion ===")
print(ans1)
print()
print("=== Q2: Most Common Complaint ===")
print(ans2)
print()
print("=== Q3: Freelancer Recommendation ===")
print(ans3)

=== Q1: Dominant Emotion ===
The dominant emotion in these freelancer reviews is dissatisfaction, with a significant number of reviews expressing negative sentiments. Examples include reviews 2, 3, 4, 13, 19, and 20, which mention wasted time and money, missed deadlines, and poor quality. Additionally, reviews 3 and 13 specifically state that the freelancer's quality was "subpar," further emphasizing the negative tone.

=== Q2: Most Common Complaint ===
The single most common complaint among the freelancer reviews is the issue of missed deadlines and subpar quality. This is evident in reviews 3, 6, 13, and 19, where the freelancer failed to meet expectations in terms of both timeliness and quality. Specifically, review 3 mentions that the freelancer "Missed deadlines repeatedly and quality was subpar."

=== Q3: Freelancer Recommendation ===
Based on these reviews, a freelancer should prioritize meeting deadlines and delivering high-quality work to avoid negative reviews. Specifically, 

In [ ]:
# T5c continued – 3-Bullet NRA Report
# Numbers are read from the printed outputs above.
nra_t5 = """
=== 3-BULLET NRA REPORT: ReviewPulse LCEL Pipeline Analysis ===

Bullet 1 (Q1: dominant emotion):
  Number : 6 reviews (2, 3, 4, 13, 19, 20) express dissatisfaction or negative sentiment.
  Reason : Clients feel a lack of control when freelancers do not communicate delays,
           leading to frustration even if final quality is acceptable.
  Action : Require freelancers to send a weekly progress update every Friday by 5 PM,
           with a clear "on track / at risk / delayed" status indicator.

Bullet 2 (Q2: most common complaint):
  Number : 4 reviews (3, 6, 13, 19) mention missed deadlines as the primary complaint.
  Reason : Deadline overruns occur because scope changes are not formally documented,
           causing misalignment on delivery time.
  Action : Implement a change‑order system: any scope addition automatically extends the
           deadline by 2 days, with client sign‑off.

Bullet 3 (Q3: freelancer recommendation):
  Number : 3 reviews (3, 6, 13) highlight that freelancers should prioritise meeting deadlines.
  Reason : Early communication builds trust and allows clients to adjust their plans,
           reducing negative sentiment even when delays occur.
  Action : Freelancers must notify the client at least 48 hours before any missed deadline
           and propose a new completion date.
"""
print(nra_t5)


=== 3-BULLET NRA REPORT: ReviewPulse LCEL Pipeline Analysis ===

Bullet 1 (Q1: dominant emotion):
  Number : 6 reviews (2, 3, 4, 13, 19, 20) express dissatisfaction or negative sentiment.
  Reason : Clients feel a lack of control when freelancers do not communicate delays,
           leading to frustration even if final quality is acceptable.
  Action : Require freelancers to send a weekly progress update every Friday by 5 PM,
           with a clear "on track / at risk / delayed" status indicator.

Bullet 2 (Q2: most common complaint):
  Number : 4 reviews (3, 6, 13, 19) mention missed deadlines as the primary complaint.
  Reason : Deadline overruns occur because scope changes are not formally documented,
           causing misalignment on delivery time.
  Action : Implement a change‑order system: any scope addition automatically extends the
           deadline by 2 days, with client sign‑off.

Bullet 3 (Q3: freelancer recommendation):
  Number : 3 reviews (3, 6, 13) highlight that f

---
## ⭐ BONUS — `.batch()`: Process Multiple Inputs Efficiently [10★]

**Objective:** Use `.batch()` to run the same LCEL chain on a list of inputs in one call.  
Compare the behaviour to calling `.invoke()` in a loop.

### Sub-tasks
- **B1:** Define 3 different questions in a list called `batch_inputs`.  
  Each element is a dict: `{"docs": docs[:10], "question": "..."}`
- **B2:** Call `full_pipeline.batch(batch_inputs)` → store as `batch_results`
- **B3:** Print all 3 results with clear labels
- **B4:** Print `len(batch_results)` and confirm it equals `len(batch_inputs)`
- **B5:** One-line comment explaining when `.batch()` is preferred over `.invoke()` in client work

**Hint:** `.batch()` signature is identical to calling `.invoke()` multiple times,  
but LangChain can use threading internally to process them faster.

In [ ]:
# --------------------------------------------------------------------
# BONUS: Use .batch() to process multiple inputs efficiently.
# GOAL: Compare batch processing to manual loop invocation.
# METHOD: Create a list of 3 input dicts, call full_pipeline.batch(inputs)
#         and print all results. Confirm count matches.
# --------------------------------------------------------------------

# B1 – Build batch input list
batch_inputs = [
    {"docs": docs[:10], "question": "Which freelance category gets the most positive reviews?"},
    {"docs": docs[:10], "question": "What is the average sentiment tone across these reviews?"},
    {"docs": docs[:10], "question": "What one change would most improve client satisfaction?"}
]

# B2 – Call .batch()
batch_results = full_pipeline.batch(batch_inputs)

# B3 – Print results
for i, res in enumerate(batch_results):
    print(f"=== Batch Result {i+1} ===")
    print(res)
    print()

# B4 – Confirm count
print(f"Inputs  : {len(batch_inputs)}")
print(f"Results : {len(batch_results)}")
print(f"Match   : {len(batch_inputs) == len(batch_results)}")

# B5 – One-line comment when to prefer .batch()
# Use .batch() when you have many independent inputs to process in parallel,
# especially in batch jobs or nightly reports, to save time and reduce API call overhead.

=== Batch Result 1 ===
Based on the review texts, the freelance category that gets the most positive reviews is the "High-end" or "Expert" category. This is evident from reviews 5 and 8, where freelancers are described as having "top-notch skills" and "exceeded all requirements," and another freelancer is praised for having "great communication and outstanding results." These reviews suggest that this category of freelancers is highly skilled and reliable.

=== Batch Result 2 ===
The average sentiment tone across these reviews is generally negative, with 6 out of 10 reviews expressing dissatisfaction or disappointment. For example, reviews 2, 3, 4, and 6 mention specific issues such as "wasted time and money," "subpar quality," and "missed deadlines repeatedly," which contribute to the overall negative tone. Only 4 reviews (1, 5, 7, and 8) express positive sentiments, with 2 of them (5 and 7) using superlatives like "top-notch" and "highly recommend."

=== Batch Result 3 ===
The most s

---
## 🏆 SCORING RUBRIC

| Task | Sub-task | Points | Criteria |
|------|----------|--------|---------|
| T1 | T1a | 4 | CSVLoader created, `.load()` called, `docs` is a list of Documents |
| T1 | T1b | 4 | All 4 prints present: len, type, page_content[:300], metadata |
| T1 | T1c | 4 | `docs[5].page_content` printed |
| T1 | T1d | 3 | NRA Number matches printed `len(docs)` exactly; Reason states causal mechanism; Action is specific |
| **T1 Total** | | **15** | |
| T2 | T2a | 4 | `ChatPromptTemplate.from_template` with `{question}` and `{context}` |
| T2 | T2b | 4 | `basic_chain = prompt_t2 \| llm \| StrOutputParser()` — exactly 3 components piped |
| T2 | T2c | 4 | `.invoke()` called with correct dict, result is a string (not AIMessage) |
| T2 | T2d | 3 | `type(basic_chain)` printed; comment/output shows `RunnableSequence` |
| **T2 Total** | | **15** | |
| T3 | T3a | 5 | `context_10` = first 10 docs joined by `"\n\n"` |
| T3 | T3b | 5 | Both chains built using `\|` operator with `StrOutputParser()` |
| T3 | T3c | 5 | `RunnableParallel({...})` invoked; both branches printed with labels |
| T3 | T3d | 5 | NRA Number from printed result; Reason = causal mechanism; Action = specific |
| **T3 Total** | | **20** | |
| T4 | T4a | 4 | `retrieve_fn` joins first 5 docs' `page_content` with `"\n---\n"` |
| T4 | T4b | 4 | Chain: `RunnablePassthrough.assign(...)  \| prompt_qa \| llm \| StrOutputParser()` |
| T4 | T4c | 4 | `.invoke()` called with `{"question": "..."}`, result printed as string |
| T4 | T4d | 3 | `type(qa_chain)` printed; one-line comment explains `RunnableSequence` |
| **T4 Total** | | **15** | |
| T5 | T5a | 4 | `extract_review_texts` correctly parses `review_text:` lines; test output shows ≥3 texts |
| T5 | T5b | 5 | `full_pipeline = RunnableLambda(extract_review_texts) \| prompt_t5 \| llm \| parser` |
| T5 | T5c | 6 | All 3 answers printed with labels; 3-bullet NRA with Numbers from printed output (not memory) |
| **T5 Total** | | **15** | |
| ★ | B1–B5 | 10★ | `batch_inputs` list of 3 dicts; `.batch()` called; all 3 results printed; count confirmed; comment present |

### NRA Deduction Rules (applies to T1d, T3d, T5 bullets)
| Violation | Deduction |
|-----------|----------|
| Number not from printed output (typed from memory/estimate) | −2 pts per bullet |
| Reason = outcome description instead of causal mechanism | −1 pt per bullet |
| Action contains hedging words (would, might, could, probably) | −1 pt per bullet |
| Internal inconsistency between bullets (contradicting recommendations) | −2 pts |

---

## 🎯 Interview Framing

> *"How would you explain LCEL to a client who asks why you rewrote their pipeline?"*

**Model Answer:**  
*"LCEL is LangChain's pipe-operator approach to building AI chains — each step is a Runnable that can be composed with `|` exactly like Unix pipes. The advantage over the older LLMChain pattern is that every LCEL chain automatically supports `.invoke()` for single inputs, `.batch()` for bulk processing, and `.stream()` for real-time output — without any extra code. For your use case, this means I can build one chain definition and deploy it three different ways depending on the traffic pattern: single-user chat via invoke, nightly bulk report via batch, and live dashboard via stream."*

---

## 📌 GitHub Commit
```
feat: Day171 - LangChain Document Loaders + LCEL [pending]
```
Repo: `Month10-LangChain-MLflow-Portfolio`